In [1]:
import xml.etree.ElementTree as ET
import copy
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

project_root = Path.cwd()

# Retention time settings
rt_start = 2.0
rt_end = 38.0
width = 0.375
num_fractions = int((rt_end - rt_start) / width)

# Helper to find a parameter by name (exact)
def find_param(step, name):
    return step.find(f"parameter[@name='{name}']")

# Helper to find a suffix parameter (name startswith 'Suffix')
def find_suffix_param(step):
    for p in step.findall("parameter"):
        if p.get('name','').startswith("Suffix"):
            return p
    return None

def process(species):
    input_batch = project_root / "data" / f"model_for_{species}_reproducibility_4.7.8.mzbatch"
    output_batch = project_root / f"fraction_batch_{species}_4.7.8.mzbatch"

    # Parse the template batch file
    tree = ET.parse(input_batch)
    root = tree.getroot()
    steps = root.findall('batchstep')

    # Extract import step (first) and pipeline templates (rest)
    import_step = steps[0]
    pipeline_templates = steps[1:]

    # Create new batch root, preserve attributes
    new_root = ET.Element('batch', root.attrib)
    new_root.append(copy.deepcopy(import_step))  # single import

    # Loop over each fraction
    for i in range(num_fractions):
        tag = f"{i+1:02d}"
        rt_min = rt_start + i * width
        rt_max = rt_min + width

        for tpl in pipeline_templates:
            step = copy.deepcopy(tpl)
            method = step.get('method','')

            if 'CropFilterModule' in method:
                sf = find_param(step, "Scan filters")
                rt = sf.find("parameter[@name='Retention time']")
                rt.find('min').text = f"{rt_min:.3f}"
                rt.find('max').text = f"{rt_max:.3f}"
                p = find_suffix_param(step)
                p.text = f"frac_{tag}"

            elif 'MassDetectionModule' in method:
                rd = find_param(step, "Raw data files")
                np = rd.find('name_pattern') or ET.SubElement(rd, 'name_pattern')
                np.text = f"*frac_{tag}"

            elif 'ChromatogramBuilderModule' in method:
                p = find_suffix_param(step)
                p.text = f"eics_{tag}"

            elif 'SmoothingModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*eics_{tag}"
                p = find_suffix_param(step)
                p.text = f"sm_{tag}"

            elif 'MinimumSearchFeatureResolverModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*sm_{tag}"
                p = find_suffix_param(step)
                p.text = f"r_{tag}"

            elif 'IsotopeGrouperModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*r_{tag}"
                p = find_param(step, "Name suffix")
                p.text = f"deiso_{tag}"

            elif 'JoinAlignerModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*deiso_{tag}"
                find_param(step, "Feature list name").text = f"Aligned feature list_{tag}"

            elif 'MultiThreadPeakFinderModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*Aligned feature list_{tag}"
                p = find_param(step, "Name suffix")
                p.text = f"gaps_{tag}"

            elif 'DuplicateFilterModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*gaps_{tag}"
                p = find_param(step, "Name suffix")
                p.text = f"dup_{tag}"

            elif 'CorrelateGroupingModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*dup_{tag}"
                p = find_suffix_param(step)
                p.text = f"metacorr_{tag}"

            elif 'IonNetworkingModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*metacorr_{tag}"

            elif 'RowsFilterModule' in method:
                find_param(step, "Name suffix").text = f"peak_{tag}"

            elif 'CSVExportModularModule' in method:
                fl = find_param(step, "Feature lists")
                np = fl.find('name_pattern') or ET.SubElement(fl, 'name_pattern')
                np.text = f"*peak_{tag}"
                fn = find_param(step, "Filename")
                cf = fn.find('current_file'); lf = fn.find('last_file')
                cf.text = cf.text.replace("frac_01", f"frac_{tag}")
                lf.text = lf.text.replace("frac_01", f"frac_{tag}")

            new_root.append(step)

    # Write out the new batch file
    ET.ElementTree(new_root).write(output_batch, encoding="UTF-8", xml_declaration=True)
    print("Written:", output_batch)


# Run both species in one go
for sp in ["84", "86"]:
    process(sp)


Written: C:\Users\bouch\OneDrive\Plocha\IOCB_Erik\PROJECTS\microfractionation\scripts\NO_batch_for_feaction_processing\batch_low_res\fraction_batch_84_4.7.8.mzbatch
Written: C:\Users\bouch\OneDrive\Plocha\IOCB_Erik\PROJECTS\microfractionation\scripts\NO_batch_for_feaction_processing\batch_low_res\fraction_batch_86_4.7.8.mzbatch
